In [1]:
import os
import sys

# 현재 작업 디렉토리 기준으로 상위 1단계 폴더를 루트로 설정
current_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(current_dir, '..'))

# sys.path에 추가 (모듈 import용)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# print("프로젝트 루트로 설정된 경로:", project_root)

In [ ]:
# 데이터 확인하기 2025.11.21 zero_count_rate > 99% 이상인 컬럼 제거 후 RandomForest 기본 모델 돌리기
import pandas as pd
import numpy  as np

from sklearn.preprocessing import StandardScaler # 데이터 전처리용

import matplotlib.pyplot as plt
import seaborn as sns

import importlib
from utils import preprocessing

# 모듈 reload
importlib.reload(preprocessing)
# importlib.reload(user_utils)

from utils.preprocessing import load_data, split_features_target, scale_data, data_split, remove_zero_columns2
from utils.user_utils    import get_model_train_eval

c:\ProgramData\anaconda3\envs\ml_dev\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# 데이터 로딩
train, test = load_data()

In [ ]:
# 데이터 할당
X_features, y_labels = split_features_target(train) # ID와 TARGET 모두 제거하고 X_train 만들기
X_test     = test.drop(columns=['ID'], axis=1) # test 데이터에서도 ID 제거 

In [ ]:
# Data 전처리 1. zero_count_rate이 99%인 컬럼 제거하기 
X_features,X_test = remove_zero_columns2(X_features,X_test,0.99)

In [7]:
# Data 전처리 2. var3 의 최소값 -99999 를 최빈값으로 변경하기
X_features['var3'] = X_features['var3'].replace(-999999, 2)

In [8]:
# 152개 컬럼의 데이터 전처리한 것 저장해 두기, 다음에 다시 테스트할때 대비
# X_features.to_csv('../data/x_train_152cols.csv')
# X_test.to_csv('../data/x_test_152cols.csv')

In [9]:
# 스케일링
X_train_scaled, X_test_scaled, scaler = scale_data(X_train=X_features, X_test=X_test)

In [ ]:
# # 레이블의 분포 확인
# cust_cnt = y_labels.value_counts()
# print(cust_cnt) # 1이 불만족 3008명, 만족이 73012

# # 불만족고객의 비율
# cust_rate = cust_cnt[1] / cust_cnt.sum()
# print(f'불만족 고객 비율: {cust_rate:.2f}')

TARGET
0    73012
1     3008
Name: count, dtype: int64
불만족 고객 비율: 0.04


In [11]:
# 학습/테스트 데이터 분리
X_train, X_val, y_train, y_val = data_split(
  X_features, 
  y_labels,
)


In [ ]:
# Model 학습, 평가
from lightgbm import LGBMClassifier

model_name = 'LightGBM_95per_basic'

# 기본값으로 우선 RF 해 보자
lgbm_clf = LGBMClassifier(
  random_state      = 0,
  n_estimators      = 100,
  num_leaves        = 15,        # 가지 수
  min_child_samples = 2
)

get_model_train_eval(lgbm_clf, model_name, X_train, X_val, y_train, y_val)

[LightGBM] [Info] Number of positive: 2406, number of negative: 58410
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.024248 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 14507
[LightGBM] [Info] Number of data points in the train set: 60816, number of used features: 319
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.039562 -> initscore=-3.189521
[LightGBM] [Info] Start training from score -3.189521
✓ 모델 저장 완료: models\LigthGBM_95per.pkl
  파일 크기: 0.18 MB
folder = c:\big20\git\big20-ML-project2-team3\SantanderCS\results
AUC: 0.8457, 정확도: 0.9577, 정밀도: 0.0980, 재현율: 0.0083, F1: 0.0153
오차행렬:
[[14556    46]
 [  597     5]]
실행 시간: 2.1099421977996826


In [ ]:
model_name = 'LightGBM_95per_HP_leaves32_child_6_class_weight'

# 기본값으로 우선 RF 해 보자
lgbm_clf = LGBMClassifier(
  random_state      = 0,
  n_estimators      = 100,
  num_leaves        = 32,        # 가지 수
  min_child_samples = 6,
  class_weight = {0:1, 1:2}
)

get_model_train_eval(lgbm_clf, model_name, X_train, X_val, y_train, y_val)

In [ ]:
model_name = 'LightGBM_95per_HP_leaves32_child18__class_weight'

# 기본값으로 우선 RF 해 보자
lgbm_clf = LGBMClassifier(
  random_state      = 0,
  n_estimators      = 100,
  num_leaves        = 32,        # 가지 수
  min_child_samples = 18,
  class_weight = {0:1, 1:2}
)

get_model_train_eval(lgbm_clf, model_name, X_train, X_val, y_train, y_val)

In [ ]:
model_name = 'LightGBM_95per_HP_ne_300_leaves32_child24__class_weight'

# 기본값으로 우선 RF 해 보자
lgbm_clf = LGBMClassifier(
  random_state      = 0,
  n_estimators      = 300,
  num_leaves        = 32,        # 가지 수
  min_child_samples = 24,
  class_weight = {0:1, 1:2}
)

get_model_train_eval(lgbm_clf, model_name, X_train, X_val, y_train, y_val)